# modeling_v7 — 물리 검증 클린 리빌드 (데이터 사전 기반)

**목표**: RMSE ~60은 일반화 바닥으로 확정(잔차·보정·편차·과도 4중 검증). v7의 목표는 RMSE 돌파가 아니라 **물리적으로 올바르고, 해석·모니터링 가능하며, 신규 lot에 견고한 실무형 모델**. 노이즈 감소로 1~2pt 덤 가능.

**데이터 사전 반영 (핵심 변경)**
- **설정값 제거**: C1(RF설정), C4/C5(가스설정), C48(가스유량 설정) — recipe만 인코딩하는 노이즈. 실측값만 사용.
- **점화/정착 분리**: Step 4의 점화 과도(C42=1, 첫 9초) vs 정착(C42=0) 별도 집계 — RF 거동(C32반사파, C18매칭, C61/C62전극, C11 Vdc)의 품질 신호.
- **C59/C60 재구성**: 상호배타 2채널 → `active = where(C59>1000, C59, C60)` + 채널 플래그.
- **드리프트 피처**: C33(PM 카운터), C25(69일 장기 드리프트), C17/C9(열 이력) — 노후도/모니터링.
- **Endpoint**: C50 Step4 종료 이벤트 (C50>0 wafer C65 소폭↑).
- Lot/WF ID 전면 배제 유지 (일반화 원칙).

**비교 기준**: v3 Test 60.51, v5 Test 60.52.

## Cell 1 — imports & 물리 검증 컬럼 정의

In [ ]:
import numpy as np, pandas as pd, json, time, warnings
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
warnings.filterwarnings("ignore")

DATA_DIR=Path("../문제1(하)"); ANS_DIR=Path("../문제1_하_answer")
OUT=Path("outputs"); OUT.mkdir(exist_ok=True)
WF_ID,TARGET,N_FOLDS = "C64","C65",5

# 실측 센서 (설정값 C1/C4/C5/C48 제외) — 데이터 사전 기준
MEAS=['C31','C32','C11','C12','C18','C61','C62','C27','C54','C56','C63',
      'C15','C16','C17','C9','C52','C57','C58','C25','C49','C46']
# Step4 점화/정착 분리 대상 RF 센서
RF=['C32','C18','C62','C61','C11','C31','C27','C63']
TE_COL="C23"

# 물리 카테고리 (피처 중요도 해석용)
PHYS_GROUP={
 "RF/플라즈마":['C31','C32','C11','C12','C18','C61','C62','C27','C54','C56','C63'],
 "가스":['C15','C16'], "온도":['C17','C9','C52'], "챔버환경":['C57','C58'],
 "드리프트/노후":['C25','C33'], "이벤트/상태":['C49','C50','C59','C60','C46']}

## Cell 2 — 데이터 로드

In [ ]:
train=pd.read_csv(DATA_DIR/"train_data.csv")
valid_X=pd.read_csv(DATA_DIR/"valid_X.csv"); test_X=pd.read_csv(DATA_DIR/"test_X.csv")
valid_Yp=pd.read_csv(DATA_DIR/"valid_Y_problem.csv"); test_Yp=pd.read_csv(DATA_DIR/"test_Y_problem.csv")
valid_Ya=pd.read_csv(ANS_DIR/"valid_Y_answer.csv").set_index("C64")["C65"]
test_Ya =pd.read_csv(ANS_DIR/"test_Y_answer.csv").set_index("C64")["C65"]
cols=[c for c in train.columns if c!=TARGET]; valid_X=valid_X[cols]; test_X=test_X[cols]
print(train.shape, valid_X.shape, test_X.shape)

## Cell 3 — 물리 검증 피처 빌더

In [ ]:
def build_v7(df, has_target=True):
    df=df.copy(); df["_dt"]=pd.to_datetime(df["C40"]); df=df.sort_values([WF_ID,"_dt"])
    # C59/C60 재구성 (상호배타 2채널 → 활성값 + 플래그)
    df["c59_act"]=np.where(df["C59"]>1000, df["C59"], df["C60"])
    df["c59_flag"]=(df["C59"]>1000).astype(float)
    g=df.groupby(WF_ID, sort=False)
    # 전역 집계 (실측 센서)
    agg=g[MEAS].agg(["mean","std","min","max","median"]); agg.columns=[f"{c}_{s}" for c,s in agg.columns]
    delta=g[MEAS].agg(lambda x:x.iloc[-1]-x.iloc[0]); delta.columns=[f"{c}_delta" for c in MEAS]
    F=pd.concat([agg,delta],axis=1)
    # 점화(C42=1) vs 정착(C42=0) — Step4
    s4=df[df["C7"]==4]
    ign=s4[s4["C42"]==1].groupby(WF_ID)[RF].mean().add_prefix("ign_")
    stl=s4[s4["C42"]==0].groupby(WF_ID)[RF].mean().add_prefix("stl_")
    F=F.join(ign).join(stl)
    for c in RF: F[f"gap_{c}"]=F.get(f"ign_{c}")-F.get(f"stl_{c}")
    # C59/C60 재구성 집계
    F["c59act_mean"]=g["c59_act"].mean(); F["c59act_std"]=g["c59_act"].std(); F["c59flag_rate"]=g["c59_flag"].mean()
    # 드리프트/노후
    F["C33_first"]=g["C33"].first(); F["C33_max"]=g["C33"].max(); F["C33_delta"]=F["C33_max"]-F["C33_first"]
    F["C25_mean"]=g["C25"].mean(); F["C25_slope"]=g["C25"].agg(lambda x:x.iloc[-1]-x.iloc[0])
    F["C17_range"]=g["C17"].max()-g["C17"].min(); F["C9_range"]=g["C9"].max()-g["C9"].min()
    # Endpoint (C50, Step4 종료행)
    s4l=s4.groupby(WF_ID).tail(1)
    F["c50_end"]=s4l.groupby(WF_ID)["C50"].max().reindex(F.index).fillna(0)
    F["c50_any"]=(s4.groupby(WF_ID)["C50"].max()>0).astype(float).reindex(F.index).fillna(0)
    # 메타 + 공정 구성
    F["n_rows"]=g.size(); F["C41_total"]=g["C41"].max()
    for lv in [1.0,4.0,5.0,6.0,7.0]:
        F[f"C7_{int(lv)}_frac"]=g["C7"].apply(lambda s,l=lv:(s==l).mean())
    F["C6_1_rate"]=g["C6"].apply(lambda s:(s=="C6_1").mean())
    F[TE_COL]=g[TE_COL].first()
    if has_target: F[TARGET]=g[TARGET].first()
    return F.reset_index()

t0=time.time()
tr_f=build_v7(train,True); va_f=build_v7(valid_X,False); te_f=build_v7(test_X,False)
FEATS=[c for c in tr_f.columns if c not in [WF_ID,TARGET,TE_COL]]
print("피처 수:", len(FEATS)+1, "| 소요 %.1fs"%(time.time()-t0))

## Cell 4 — C23 out-of-fold 타깃인코딩 헬퍼

In [ ]:
def te_fit(frame, col, target, m=20):
    prior=frame[target].mean()
    agg=frame.groupby(col)[target].agg(["mean","count"])
    return (agg["mean"]*agg["count"]+prior*m)/(agg["count"]+m), prior

## Cell 5 — GroupKFold 학습 (v3 최적 파라미터 재사용)

In [ ]:
PARAMS=dict(objective="regression",metric="rmse",boosting_type="gbdt",verbosity=-1,
    random_state=42,n_jobs=-1,n_estimators=3000,
    learning_rate=0.00576,num_leaves=189,max_depth=10,min_child_samples=14,
    subsample=0.967,subsample_freq=6,colsample_bytree=0.655,
    reg_alpha=4.256,reg_lambda=0.003,min_split_gain=0.758)

y=tr_f[TARGET].values; gr=tr_f[WF_ID].astype("category").cat.codes.values
gkf=GroupKFold(N_FOLDS); USE=FEATS+[TE_COL+"_te"]
oof=np.zeros(len(tr_f)); va_pred=np.zeros(len(va_f)); te_pred=np.zeros(len(te_f))
last_model=None
for k,(tri,vai) in enumerate(gkf.split(tr_f,y,gr)):
    tf=tr_f.iloc[tri].copy(); vf=tr_f.iloc[vai].copy()
    enc,prior=te_fit(tf,TE_COL,TARGET)
    tf[TE_COL+"_te"]=tf[TE_COL].map(enc).fillna(prior); vf[TE_COL+"_te"]=vf[TE_COL].map(enc).fillna(prior)
    va_k=va_f.copy(); te_k=te_f.copy()
    va_k[TE_COL+"_te"]=va_k[TE_COL].map(enc).fillna(prior); te_k[TE_COL+"_te"]=te_k[TE_COL].map(enc).fillna(prior)
    m=lgb.LGBMRegressor(**PARAMS)
    m.fit(tf[USE],tf[TARGET],eval_set=[(vf[USE],vf[TARGET])],
          callbacks=[lgb.early_stopping(100,verbose=False),lgb.log_evaluation(0)])
    oof[vai]=m.predict(vf[USE]); va_pred+=m.predict(va_k[USE])/N_FOLDS; te_pred+=m.predict(te_k[USE])/N_FOLDS
    last_model=m; print(f"fold{k+1} best_iter={m.best_iteration_}")

## Cell 6 — 평가

In [ ]:
oof_rmse=np.sqrt(mean_squared_error(y,oof))
va_s=pd.Series(va_pred,index=va_f[WF_ID]); te_s=pd.Series(te_pred,index=te_f[WF_ID])
valid_rmse=np.sqrt(mean_squared_error(valid_Ya.loc[va_s.index],va_s))
test_rmse =np.sqrt(mean_squared_error(test_Ya.loc[te_s.index],te_s))
print(f"CV OOF RMSE : {oof_rmse:.4f}")
print(f"Valid  RMSE : {valid_rmse:.4f}   (v3 62.31 / v5 61.38)")
print(f"Test   RMSE : {test_rmse:.4f}   (v3 60.51 / v5 60.52)")

## Cell 7 — 물리 그룹별 피처 중요도 (해석/모니터링)

In [ ]:
imp=pd.Series(last_model.booster_.feature_importance("gain"),index=USE).sort_values(ascending=False)
print("=== Top 15 피처 ==="); print(imp.head(15).round(0))
# 물리 그룹별 중요도 합산
def grp(f):
    for gname,cols in PHYS_GROUP.items():
        if any(f.startswith(c+"_") or f.startswith("ign_"+c) or f.startswith("stl_"+c) or f.startswith("gap_"+c) for c in cols): return gname
    if f.startswith(TE_COL): return "Recipe(C23)"
    if f.startswith("C7_") or f.startswith("C6"): return "공정구성"
    if f.startswith("c59"): return "이벤트/상태"
    if f.startswith("c50"): return "이벤트/상태"
    return "기타"
gi=imp.groupby(imp.index.map(grp)).sum().sort_values(ascending=False)
print("\n=== 물리 그룹별 중요도(gain 합) ==="); print((gi/gi.sum()*100).round(1))

## Cell 8 — 제출 저장

In [ ]:
vs=valid_Yp.copy(); vs["C65"]=vs["C64"].map(va_s); vs.to_csv(OUT/"valid_Y_submit.csv",index=False)
ts=test_Yp.copy();  ts["C65"]=ts["C64"].map(te_s); ts.to_csv(OUT/"test_Y_submit.csv",index=False)
json.dump({"oof":round(float(oof_rmse),4),"valid":round(float(valid_rmse),4),"test":round(float(test_rmse),4),
           "n_features":len(USE)}, open(OUT/"results.json","w"), indent=2, ensure_ascii=False)
print("저장 완료 | 피처 수:", len(USE))